In [1]:
import pandas as pd

In [2]:
data_folder = "../data/"
movies_file_path = data_folder + "movies.csv"
ratings_file_path = data_folder + "ratings.csv"

movies = pd.read_csv(movies_file_path,index_col=None)
ratings = pd.read_csv(ratings_file_path,index_col=None)

In [3]:
all_genres = movies['genres'].str.split('|').explode()
genre_counts = all_genres.value_counts()
genre_counts

genres
Drama                 25606
Comedy                16870
Thriller               8654
Romance                7719
Action                 7348
Horror                 5989
Documentary            5605
Crime                  5319
(no genres listed)     5062
Adventure              4145
Sci-Fi                 3595
Children               2935
Animation              2929
Mystery                2925
Fantasy                2731
War                    1874
Western                1399
Musical                1054
Film-Noir               353
IMAX                    195
Name: count, dtype: int64

In [4]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [21]:
movies[movies['title'] == 'Aladdin (1992)'][['movieId','title','genres']]

,movieId,title,genres
580,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical
22286,114240,Aladdin (1992),Adventure|Animation|Children|Comedy|Fantasy


In [6]:
movies['title'].value_counts()

title
The Forest (2016)                  2
Blockbuster (2017)                 2
Hostage (2005)                     2
Delirium (2018)                    2
Free Fall (2014)                   2
                                  ..
Punk's Dead: SLC Punk! 2 (2014)    1
Chinese Hercules (1973)            1
Q (2011)                           1
In the Fog (V tumane) (2012)       1
The Phynx (1970)                   1
Name: count, Length: 62325, dtype: int64

In [7]:
movies[movies['movieId'].isin([147458, 172035,131078,197671])] ## Checking if we are properly mapping in below step

,movieId,title,genres
28165,131078,Benjamin Blümchen - Seine schönsten Abenteuer ...,(no genres listed)
35314,147458,The Blancheville Monster (1963),(no genres listed)
46236,172035,Holiday For Lovers (1959),(no genres listed)
58004,197671,Hank Aaron: Chasing the Dream (1995),(no genres listed)


In [8]:
movies = movies[movies['genres'] != '(no genres listed)']

In [9]:
# data cleaning
# merge all movies with a global genre list
merged_movies_df = (
    movies.groupby('title', as_index=False)
    .agg({
        'movieId': 'first',  # Pick first movieId (you could also choose most rated if needed)
        'genres': lambda x: '|'.join(sorted(set('|'.join(x).split('|'))))
    })
)

In [10]:
genres_one_hot = merged_movies_df['genres'].str.get_dummies(sep='|')
genre_features = genres_one_hot.values
genre_features.shape

(57278, 19)

In [11]:
merged_movies_df = merged_movies_df.reset_index(drop=True)

In [22]:
merged_movies_df[merged_movies_df['title'] == 'Aladdin (1992)'][['movieId','title','genres']]

,movieId,title,genres
2209,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Fantasy|Mu...


In [12]:
title_to_main_id = dict(zip(movies['title'], movies['movieId']))

In [13]:
# Map all old movieIds to corresponding main movieId by title
old_to_main_id = {}

for title, main_id in title_to_main_id.items():
    # Find all original movieIds for this title
    old_ids = movies[movies['title'] == title]['movieId'].tolist()
    for oid in old_ids:
        old_to_main_id[oid] = main_id

In [15]:
ratings['movieId'] = ratings['movieId'].map(old_to_main_id)
ratings = ratings.dropna(subset=['movieId'])
ratings['movieId'] = ratings['movieId'].astype(int)

In [16]:
missing_ids = set(ratings['movieId'].unique()) - set(old_to_main_id.keys())
print(f"Number of missing movieIds in mapping: {len(missing_ids)}")

Number of missing movieIds in mapping: 0


In [17]:
# print(missing_ids)

In [40]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_movies(title, num_recommendations=5):
    if title not in indices:
        print("Movie not found in dataset.")
        return pd.DataFrame()

    # Get index of target movie
    idx = indices[title]

    # Get its genre vector
    target_vector = genre_features[idx].reshape(1, -1)
    # print(target_vector.shape,"<<<>>>>",genre_features.shape)

    # Compute similarity only for this vector
    sim_scores = cosine_similarity(target_vector, genre_features)[0]

    # Pair with indices
    sim_scores = list(enumerate(sim_scores))

    # Sort by similarity score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Skip itself (first entry)
    sim_scores = sim_scores[1:num_recommendations+1]

    # Get indices of similar movies
    movie_indices = [i[0] for i in sim_scores]

    return movies.iloc[movie_indices][['title', 'genres']]

In [41]:
# Map from movie title to index in dataframe
indices = pd.Series(merged_movies_df.index, index=merged_movies_df['title']).drop_duplicates()

In [42]:
recommend_movies("Aladdin (1992)")

,title,genres
9489,Man Trouble (1992),Comedy|Romance
15000,"Power of Kangwon Province, The (Kangwon-do ui ...",Drama
1915,Gremlins 2: The New Batch (1990),Comedy|Horror
2321,Rocky V (1990),Action|Drama
3517,Loser (2000),Comedy|Romance


In [43]:
user_rating_counts = ratings.groupby('userId').size()
active_users = user_rating_counts[user_rating_counts >= 20].index.tolist()

In [44]:
user_id = active_users[0]
print(f"Chosen user: {user_id}, number of ratings: {user_rating_counts[user_id]}")

Chosen user: 1, number of ratings: 70


In [45]:
user_likes = ratings[(ratings['userId'] == 1) & (ratings['rating'] >= 4.0)]
liked_movie_ids = user_likes['movieId'].tolist()
print(f"User 1 liked {len(liked_movie_ids)} movies.")

User 1 liked 39 movies.


In [46]:
merged_movies_df.head()

,title,movieId,genres
0,"""BLOW THE NIGHT!"" Let's Spend the Night Togeth...",208297,Documentary|Drama
1,"""Great Performances"" Cats (1998)",51372,Musical
2,#1 Cheerleader Camp (2010),136604,Comedy|Drama
3,#Captured (2017),183901,Horror
4,#Female Pleasure (2018),195955,Documentary


In [49]:
def evaluate_user(user_id, k=5):
    # Get liked movies for the user
    user_likes = ratings[(ratings['userId'] == user_id) & (ratings['rating'] >= 4.0)]
    liked_movie_ids = user_likes['movieId'].tolist()
    # print(len(user_likes),"<<>>>",len(liked_movie_ids))

    # Skip if not enough liked movies
    if len(liked_movie_ids) < 5:
        return None

    hits = 0
    total_recs = 0

    for movie_id in liked_movie_ids:
        # Get title
        title = merged_movies_df[merged_movies_df['movieId'] == movie_id]['title'].values
        if title.size <= 0: 
            print(f"Ignoring movie id {movie_id}")
            continue
            
        title = title[0]
        try:
            # Generate recommendations
            recs_df = recommend_movies(title, num_recommendations=k)
            rec_titles = recs_df['title'].tolist()

            # print(title,"<><>>",rec_titles)

            # Map recommended titles to movie IDs
            rec_movie_ids = merged_movies_df[merged_movies_df['title'].isin(rec_titles)]['movieId'].tolist()

            # Count hits (recommended & liked)
            hits += len(set(rec_movie_ids) & set(liked_movie_ids))
            total_recs += k

        except IndexError:
            continue  # Movie not found
        except Exception:
            print("===========",title)
            # import traceback
            # print(traceback.print_exc())

    precision = hits / total_recs if total_recs else 0
    recall = hits / len(liked_movie_ids) if liked_movie_ids else 0

    return precision, recall

In [50]:
precision, recall = evaluate_user(5, k=5)

print(f"Precision@5: {precision:.3f}")
print(f"Recall@5: {recall:.3f}")

Ignoring movie id 114240
Precision@5: 0.021
Recall@5: 0.103


In [51]:
# Select all active users with at least 20 ratings
user_rating_counts = ratings.groupby('userId').size()
active_users = user_rating_counts[user_rating_counts >= 20].index.tolist()

# Pick first 10 users for evaluation
sample_users = active_users[:10]

print("Sample user IDs:", sample_users)

Sample user IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [52]:
precision_list = []
recall_list = []

for uid in sample_users:
    result = evaluate_user(uid, k=5)
    print(result)
    if result:
        precision_list.append(result[0])
        recall_list.append(result[1])

(0.0, 0.0)
(0.017699115044247787, 0.08849557522123894)
Ignoring movie id 204982
(0.02247191011235955, 0.11204481792717087)
Ignoring movie id 204982
(0.001941747572815534, 0.009615384615384616)
Ignoring movie id 114240
(0.021052631578947368, 0.10344827586206896)
(0.01, 0.05)
Ignoring movie id 114240
(0.0, 0.0)
Ignoring movie id 114240
(0.041758241758241756, 0.20652173913043478)
Ignoring movie id 114240
(0.017094017094017096, 0.0847457627118644)
(0.0, 0.0)


In [53]:
import numpy as np

mean_precision = np.mean(precision_list)
mean_recall = np.mean(recall_list)

print(f"Average Precision@5 over {len(precision_list)} users: {mean_precision:.3f}")
print(f"Average Recall@5 over {len(recall_list)} users: {mean_recall:.3f}")

Average Precision@5 over 10 users: 0.013
Average Recall@5 over 10 users: 0.065


Baseline 
--------
- precision ~1%
- recall ~6.5%

In [55]:
merged_movies_df['combined_text'] = merged_movies_df['title'] + " " + merged_movies_df['genres']
merged_movies_df[['title', 'combined_text']].head()


,title,combined_text
0,"""BLOW THE NIGHT!"" Let's Spend the Night Togeth...","""BLOW THE NIGHT!"" Let's Spend the Night Togeth..."
1,"""Great Performances"" Cats (1998)","""Great Performances"" Cats (1998) Musical"
2,#1 Cheerleader Camp (2010),#1 Cheerleader Camp (2010) Comedy|Drama
3,#Captured (2017),#Captured (2017) Horror
4,#Female Pleasure (2018),#Female Pleasure (2018) Documentary


In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Fit and transform on combined text
tfidf_matrix = tfidf.fit_transform(merged_movies_df['combined_text'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix shape: (57278, 32259)


In [57]:
from sklearn.metrics.pairwise import linear_kernel

def recommend_movies(title, num_recommendations=5):
    if title not in indices:
        print("Movie not found.")
        return pd.DataFrame()

    idx = indices[title]
    cosine_sim = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()

    # Get scores with indices
    sim_scores = list(enumerate(cosine_sim))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:num_recommendations+1]  # skip self

    movie_indices = [i[0] for i in sim_scores]
    return merged_movies_df.iloc[movie_indices][['title', 'genres']]

In [59]:
recommend_movies("Toy Story (1995)")

,title,genres
52405,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
52407,Toy Story 4 (2019),Adventure|Animation|Children|Comedy
52406,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX
52412,Toy Story of Terror (2013),Animation|Children|Comedy
52409,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy


In [ ]:
def evaluate_user(user_id, k=5):
    # Get liked movies for the user
    user_likes = ratings[(ratings['userId'] == user_id) & (ratings['rating'] >= 4.0)]
    liked_movie_ids = user_likes['movieId'].tolist()
    # print(len(user_likes),"<<>>>",len(liked_movie_ids))

    # Skip if not enough liked movies
    if len(liked_movie_ids) < 5:
        return None

    hits = 0
    total_recs = 0

    for movie_id in liked_movie_ids:
        # Get title
        title = merged_movies_df[merged_movies_df['movieId'] == movie_id]['title'].values
        if title.size <= 0: 
            print(f"Ignoring movie id {movie_id}")
            continue
            
        title = title[0]
        try:
            # Generate recommendations
            recs_df = recommend_movies(title, num_recommendations=k)
            rec_titles = recs_df['title'].tolist()

            # print(title,"<><>>",rec_titles)

            # Map recommended titles to movie IDs
            rec_movie_ids = merged_movies_df[merged_movies_df['title'].isin(rec_titles)]['movieId'].tolist()

            # Count hits (recommended & liked)
            hits += len(set(rec_movie_ids) & set(liked_movie_ids))
            total_recs += k

        except IndexError:
            continue  # Movie not found
        except Exception:
            print("===========",title)
            # import traceback
            # print(traceback.print_exc())

    precision = hits / total_recs if total_recs else 0
    recall = hits / len(liked_movie_ids) if liked_movie_ids else 0

    return precision, recall

In [60]:
precision_list = []
recall_list = []

for uid in sample_users:
    result = evaluate_user(uid, k=5)
    if result:
        precision_list.append(result[0])
        recall_list.append(result[1])

Ignoring movie id 204982
Ignoring movie id 204982
Ignoring movie id 114240
Ignoring movie id 114240
Ignoring movie id 114240
Ignoring movie id 114240


In [61]:
import numpy as np

mean_precision = np.mean(precision_list)
mean_recall = np.mean(recall_list)

print(f"Average Precision@5 over {len(precision_list)} users: {mean_precision:.3f}")
print(f"Average Recall@5 over {len(recall_list)} users: {mean_recall:.3f}")

Average Precision@5 over 10 users: 0.045
Average Recall@5 over 10 users: 0.221


Improvements
------------
- Precision : 4.5%
- Recall : 22%